In [ ]:
# ==========================================================
#   ML EXAM CHEAT SHEET — Preprocessing + Models
#   (Simple, exam-friendly syntax — sklearn style)
# ==========================================================

# ---------- 1. IMPORTS ----------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder, PowerTransformer
)
from sklearn.feature_selection import SelectKBest, f_classif, f_regression, RFE
from sklearn.decomposition import PCA

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.cluster import KMeans, DBSCAN

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    classification_report, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)

# ==========================================================
#   2. LOAD DATA
# ==========================================================
df = pd.read_csv("data.csv")

print(df.head())
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.dtypes)

# ==========================================================
#   3. EDA — Univariate / Bivariate
# ==========================================================
# Univariate
sns.histplot(df['Age'], kde=True)
plt.show()

sns.countplot(x='Survived', data=df)
plt.show()

# Bivariate
sns.boxplot(x='Survived', y='Age', data=df)
plt.show()

sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

# ==========================================================
#   4. DROP UNNECESSARY COLUMNS
# ==========================================================
df = df.drop(columns=['PassengerId', 'Name', 'Ticket'])

# ==========================================================
#   5. FEATURE / TARGET SPLIT
# ==========================================================
X = df.drop(columns=['Survived'])
y = df['Survived']

# ==========================================================
#   6. TRAIN-TEST SPLIT   (ALWAYS BEFORE Missing/Outlier/Encoding/Scaling!)
# ==========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # stratify only for classification
)

# ==========================================================
#   7. MISSING VALUE HANDLING
# ==========================================================
# ---- Manual (no pipeline) ----
imputer = SimpleImputer(strategy='median')             # 'mean','median','most_frequent','constant'
X_train[['Age']] = imputer.fit_transform(X_train[['Age']])
X_test[['Age']]  = imputer.transform(X_test[['Age']])  # ONLY transform on test!

cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[['Embarked']] = cat_imputer.fit_transform(X_train[['Embarked']])
X_test[['Embarked']]  = cat_imputer.transform(X_test[['Embarked']])

const_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')

# ---- KNN Imputer (uses neighbours' values) ----
knn_imputer = KNNImputer(n_neighbors=5)
X_train_num = knn_imputer.fit_transform(X_train[['Age', 'Fare']])

# ==========================================================
#   8. OUTLIER DETECTION & HANDLING
# ==========================================================
# ---- IQR Method ----
Q1 = X_train['Fare'].quantile(0.25)
Q3 = X_train['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
X_train['Fare'] = np.where(X_train['Fare'] > upper, upper,
                    np.where(X_train['Fare'] < lower, lower, X_train['Fare']))  # capping

# ---- Z-score Method ----
from scipy import stats
z_scores = np.abs(stats.zscore(X_train['Fare']))
X_train_no_outlier = X_train[z_scores < 3]

# ==========================================================
#   9. ENCODING (Categorical -> Numeric)
# ==========================================================
# ---- Label Encoding (target / binary column) ----
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)

# ---- Ordinal Encoding (ordered categories) ----
oe = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
X_train[['Level']] = oe.fit_transform(X_train[['Level']])
X_test[['Level']]  = oe.transform(X_test[['Level']])

# ---- One-Hot Encoding (nominal categories) ----
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
X_train_ohe = ohe.fit_transform(X_train[['Sex', 'Embarked']])
X_test_ohe  = ohe.transform(X_test[['Sex', 'Embarked']])

# ==========================================================
#   10. FEATURE SCALING
# ==========================================================
ss = StandardScaler()                     # mean=0, std=1
X_train[['Age', 'Fare']] = ss.fit_transform(X_train[['Age', 'Fare']])
X_test[['Age', 'Fare']]  = ss.transform(X_test[['Age', 'Fare']])

mm = MinMaxScaler(feature_range=(0, 1))   # scales to [0,1]
rb = RobustScaler()                       # uses median/IQR, good for outliers
pt = PowerTransformer(method='yeo-johnson')  # fixes skewness

# ==========================================================
#   11. FEATURE ENGINEERING (example)
# ==========================================================
X_train['FamilySize'] = X_train['SibSp'] + X_train['Parch'] + 1
X_train['IsAlone'] = (X_train['FamilySize'] == 1).astype(int)

# ==========================================================
#   12. FEATURE SELECTION
# ==========================================================
# ---- SelectKBest ----
selector = SelectKBest(score_func=f_classif, k=5)   # f_regression for regression
X_train_new = selector.fit_transform(X_train_num, y_train)
selected_cols = selector.get_support()

# ---- RFE (Recursive Feature Elimination) ----
rfe = RFE(estimator=LogisticRegression(), n_features_to_select=5)
X_train_rfe = rfe.fit_transform(X_train_num, y_train)
print(rfe.ranking_)

# ==========================================================
#   13. MULTICOLLINEARITY (VIF)
# ==========================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data['feature'] = X_train_num_df.columns          # numeric dataframe
vif_data['VIF'] = [variance_inflation_factor(X_train_num_df.values, i)
                    for i in range(X_train_num_df.shape[1])]
print(vif_data)   # VIF > 5 (or 10) => drop that feature

# ==========================================================
#   14. CLASS IMBALANCE (SMOTE)
# ==========================================================
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_num, y_train)

# ==========================================================
#   15. COLUMNTRANSFORMER  (numeric + categorical together)
# ==========================================================
num_cols = ['Age', 'Fare', 'FamilySize']
cat_cols = ['Sex', 'Embarked']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
], remainder='drop')          # remainder='passthrough' to keep other cols

# ==========================================================
#   16. FULL PIPELINE  (preprocessing + model)
# ==========================================================
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

full_pipe.fit(X_train, y_train)
y_pred = full_pipe.predict(X_test)
print("Pipeline Accuracy:", accuracy_score(y_test, y_pred))

# ==========================================================
#   17. CROSS VALIDATION
# ==========================================================
scores = cross_val_score(full_pipe, X_train, y_train, cv=5, scoring='accuracy')
print("CV Scores:", scores)
print("Mean CV Score:", scores.mean())

# ==========================================================
#   18. EVALUATION METRICS
# ==========================================================
# ---- Classification ----
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))

# ---- Regression ----
# print("MSE:", mean_squared_error(y_test, y_pred))
# print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
# print("MAE:", mean_absolute_error(y_test, y_pred))
# print("R2 Score:", r2_score(y_test, y_pred))


# ==========================================================
# ==========================================================
#                         MODELS
# ==========================================================
# ==========================================================

# ----------------------------------------------------------
#   19. SIMPLE LINEAR REGRESSION  (1 feature)
# ----------------------------------------------------------
lr = LinearRegression()
lr.fit(X_train[['Fare']], y_train)          # y_train must be numeric target
y_pred = lr.predict(X_test[['Fare']])
print("Coef:", lr.coef_, "Intercept:", lr.intercept_)
print("R2:", r2_score(y_test, y_pred))

# ----------------------------------------------------------
#   20. MULTIPLE LINEAR REGRESSION  (many features)
# ----------------------------------------------------------
mlr = LinearRegression()
mlr.fit(X_train, y_train)
y_pred = mlr.predict(X_test)
print("Coefficients:", mlr.coef_)
print("R2:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# ----------------------------------------------------------
#   21. OLS (Ordinary Least Squares) — statsmodels (gives p-values, summary)
# ----------------------------------------------------------
import statsmodels.api as sm

X_train_sm = sm.add_constant(X_train)        # adds intercept column
ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

# ----------------------------------------------------------
#   22. LOGISTIC REGRESSION (classification)
# ----------------------------------------------------------
log_reg = LogisticRegression(
    penalty='l2', C=1.0, solver='lbfgs', max_iter=1000, random_state=42
)
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]   # probability of class 1
print("Accuracy:", accuracy_score(y_test, y_pred))

# ----------------------------------------------------------
#   23. K-NEAREST NEIGHBORS (KNN)
# ----------------------------------------------------------
knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)  # p=2 -> Euclidean
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print("KNN Accuracy:", accuracy_score(y_test, y_pred))

# KNN Regressor
knn_reg = KNeighborsRegressor(n_neighbors=5)

# ----------------------------------------------------------
#   24. GRIDSEARCHCV  (Hyperparameter tuning — example with KNN)
# ----------------------------------------------------------
param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

# ----------------------------------------------------------
#   25. DECISION TREE
# ----------------------------------------------------------
dt = DecisionTreeClassifier(
    criterion='gini', max_depth=5, min_samples_split=2,
    min_samples_leaf=1, random_state=42
)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))
print("Feature Importances:", dt.feature_importances_)

# Decision Tree Regressor
dt_reg = DecisionTreeRegressor(max_depth=5, random_state=42)

# Visualize tree
from sklearn.tree import plot_tree
plt.figure(figsize=(15, 8))
plot_tree(dt, feature_names=X_train.columns, class_names=['0', '1'], filled=True)
plt.show()

# ----------------------------------------------------------
#   26. RANDOM FOREST
# ----------------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=100, criterion='gini', max_depth=None,
    min_samples_split=2, max_features='sqrt', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print("Feature Importances:", rf.feature_importances_)

# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)

# ----------------------------------------------------------
#   27. SUPPORT VECTOR MACHINE (SVM)
# ----------------------------------------------------------
svc = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
# kernel options: 'linear', 'poly', 'rbf', 'sigmoid'
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred))

# SVM Regressor
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)

# ----------------------------------------------------------
#   28. K-MEANS CLUSTERING (unsupervised)
# ----------------------------------------------------------
# Elbow method to find best k
inertia = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_train_num)
    inertia.append(km.inertia_)
plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel('k'); plt.ylabel('Inertia'); plt.show()

kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels = kmeans.fit_predict(X_train_num)
print("Cluster Centers:", kmeans.cluster_centers_)
print("Inertia:", kmeans.inertia_)

from sklearn.metrics import silhouette_score
print("Silhouette Score:", silhouette_score(X_train_num, labels))

# ----------------------------------------------------------
#   29. DBSCAN CLUSTERING (density based, detects outliers as -1)
# ----------------------------------------------------------
dbscan = DBSCAN(eps=0.5, min_samples=5, metric='euclidean')
labels = dbscan.fit_predict(X_train_num)
print("Unique Clusters:", set(labels))     # -1 = noise/outlier
print("Number of clusters:", len(set(labels)) - (1 if -1 in labels else 0))

# ----------------------------------------------------------
#   30. PCA (Principal Component Analysis — dimensionality reduction)
# ----------------------------------------------------------
pca = PCA(n_components=2, random_state=42)     # or n_components=0.95 for 95% variance
X_train_pca = pca.fit_transform(X_train_num)
print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Total Variance Captured:", sum(pca.explained_variance_ratio_))

plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

# ==========================================================
#   31. SAVE / LOAD MODEL
# ==========================================================
import joblib
joblib.dump(full_pipe, 'model.pkl')
loaded_model = joblib.load('model.pkl')